<a href="https://colab.research.google.com/github/prasertrak/Advanced-Data-Engineering-and-Applied-Analytics/blob/main/Part4_datetime_incremental_loading.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part 4 — Date & Time Handling
## Order Fulfillment Analytics Pipeline


### Learning Objectives
- Parse different date formats
- Handle invalid dates
- Calculate delivery duration
- Understand watermarking
- Understand late events
- Simulate incremental loading


In [ ]:
import pandas as pd
from datetime import datetime, timedelta

## 1. Create Mock Orders Dataset

In [ ]:
data = {
    "order_id": [
        "ORD001","ORD002","ORD003","ORD004","ORD005"
    ],
    "order_date": [
        "2026-05-01",
        "03/05/2026",
        "2026/05/03",
        "2026-13-01",
        "2026-05-05"
    ],
    "delivery_date": [
        "2026-05-03",
        "2026-05-08",
        "2026-05-04",
        "2026-05-09",
        None
    ],
    "updated_at": [
        "2026-05-10 10:00:00",
        "2026-05-10 10:05:00",
        "2026-05-10 10:10:00",
        "2026-05-10 10:15:00",
        "2026-05-10 10:20:00"
    ]
}

df = pd.DataFrame(data)

df

,order_id,order_date,delivery_date,updated_at
0,ORD001,2026-05-01,2026-05-03,2026-05-10 10:00:00
1,ORD002,03/05/2026,2026-05-08,2026-05-10 10:05:00
2,ORD003,2026/05/03,2026-05-04,2026-05-10 10:10:00
3,ORD004,2026-13-01,2026-05-09,2026-05-10 10:15:00
4,ORD005,2026-05-05,None,2026-05-10 10:20:00


## 2. Convert order_date to Datetime

In [ ]:
df["order_date"] = pd.to_datetime(
    df["order_date"],
    errors="coerce"
)

df

,order_id,order_date,delivery_date,updated_at
0,ORD001,2026-05-01,2026-05-03,2026-05-10 10:00:00
1,ORD002,NaT,2026-05-08,2026-05-10 10:05:00
2,ORD003,NaT,2026-05-04,2026-05-10 10:10:00
3,ORD004,NaT,2026-05-09,2026-05-10 10:15:00
4,ORD005,2026-05-05,None,2026-05-10 10:20:00


## 3. Detect Invalid Dates

In [ ]:
invalid_dates = df[
    df["order_date"].isnull()
]

invalid_dates

,order_id,order_date,delivery_date,updated_at
1,ORD002,NaT,2026-05-08,2026-05-10 10:05:00
2,ORD003,NaT,2026-05-04,2026-05-10 10:10:00
3,ORD004,NaT,2026-05-09,2026-05-10 10:15:00


## 4. Convert delivery_date to Datetime

In [ ]:
df["delivery_date"] = pd.to_datetime(
    df["delivery_date"],
    errors="coerce"
)

df

,order_id,order_date,delivery_date,updated_at
0,ORD001,2026-05-01,2026-05-03,2026-05-10 10:00:00
1,ORD002,NaT,2026-05-08,2026-05-10 10:05:00
2,ORD003,NaT,2026-05-04,2026-05-10 10:10:00
3,ORD004,NaT,2026-05-09,2026-05-10 10:15:00
4,ORD005,2026-05-05,NaT,2026-05-10 10:20:00


## 5. Calculate delivery_days

In [ ]:
df["delivery_days"] = (
    df["delivery_date"]
    - df["order_date"]
).dt.days

df

,order_id,order_date,delivery_date,updated_at,delivery_days
0,ORD001,2026-05-01,2026-05-03,2026-05-10 10:00:00,2.0
1,ORD002,NaT,2026-05-08,2026-05-10 10:05:00,NaN
2,ORD003,NaT,2026-05-04,2026-05-10 10:10:00,NaN
3,ORD004,NaT,2026-05-09,2026-05-10 10:15:00,NaN
4,ORD005,2026-05-05,NaT,2026-05-10 10:20:00,NaN


## 6. Create SLA Status

In [ ]:
from numpy import nan
df["sla_status"] = "ON_TIME"

df.loc[
    df["delivery_days"] > 7,
    "sla_status"
] = "LATE"

df

,order_id,order_date,delivery_date,updated_at,delivery_days,sla_status
0,ORD001,2026-05-01,2026-05-03,2026-05-10 10:00:00,2.0,ON_TIME
1,ORD002,NaT,2026-05-08,2026-05-10 10:05:00,NaN,ON_TIME
2,ORD003,NaT,2026-05-04,2026-05-10 10:10:00,NaN,ON_TIME
3,ORD004,NaT,2026-05-09,2026-05-10 10:15:00,NaN,ON_TIME
4,ORD005,2026-05-05,NaT,2026-05-10 10:20:00,NaN,ON_TIME


In [ ]:
from numpy import nan
df["sla_status"] = "ON_TIME"

df.loc[
    df["delivery_days"].isna(),
    "sla_status"
] = "UNKNOWN"

df

,order_id,order_date,delivery_date,updated_at,delivery_days,sla_status
0,ORD001,2026-05-01,2026-05-03,2026-05-10 10:00:00,2.0,ON_TIME
1,ORD002,NaT,2026-05-08,2026-05-10 10:05:00,NaN,UNKNOWN
2,ORD003,NaT,2026-05-04,2026-05-10 10:10:00,NaN,UNKNOWN
3,ORD004,NaT,2026-05-09,2026-05-10 10:15:00,NaN,UNKNOWN
4,ORD005,2026-05-05,NaT,2026-05-10 10:20:00,NaN,UNKNOWN


## 7. Convert updated_at to Timestamp

In [ ]:
df

,order_id,order_date,delivery_date,updated_at,delivery_days,sla_status
0,ORD001,2026-05-01,2026-05-03,2026-05-10 10:00:00,2.0,ON_TIME
1,ORD002,NaT,2026-05-08,2026-05-10 10:05:00,NaN,UNKNOWN
2,ORD003,NaT,2026-05-04,2026-05-10 10:10:00,NaN,UNKNOWN
3,ORD004,NaT,2026-05-09,2026-05-10 10:15:00,NaN,UNKNOWN
4,ORD005,2026-05-05,NaT,2026-05-10 10:20:00,NaN,UNKNOWN


In [ ]:
df["updated_at"] = pd.to_datetime(
    df["updated_at"]
)

df

,order_id,order_date,delivery_date,updated_at,delivery_days,sla_status
0,ORD001,2026-05-01,2026-05-03,2026-05-10 10:00:00,2.0,ON_TIME
1,ORD002,NaT,2026-05-08,2026-05-10 10:05:00,NaN,UNKNOWN
2,ORD003,NaT,2026-05-04,2026-05-10 10:10:00,NaN,UNKNOWN
3,ORD004,NaT,2026-05-09,2026-05-10 10:15:00,NaN,UNKNOWN
4,ORD005,2026-05-05,NaT,2026-05-10 10:20:00,NaN,UNKNOWN


## 8. Simulate Watermark

In [ ]:
last_watermark = pd.Timestamp(
    "2026-05-10 10:07:00"
)

# สร้าง dataframe ใหม่ที่เอาเฉพาะรายการที่ มีค่า updated_at มากกว่าค่า last_watermark
incremental_df = df[
    df["updated_at"] > last_watermark
]

incremental_df

,order_id,order_date,delivery_date,updated_at,delivery_days,sla_status
2,ORD003,NaT,2026-05-04,2026-05-10 10:10:00,NaN,UNKNOWN
3,ORD004,NaT,2026-05-09,2026-05-10 10:15:00,NaN,UNKNOWN
4,ORD005,2026-05-05,NaT,2026-05-10 10:20:00,NaN,UNKNOWN


## 9. Simulate Overlap Window

In [ ]:
df

,order_id,order_date,delivery_date,updated_at,delivery_days,sla_status
0,ORD001,2026-05-01,2026-05-03,2026-05-10 10:00:00,2.0,ON_TIME
1,ORD002,NaT,2026-05-08,2026-05-10 10:05:00,NaN,UNKNOWN
2,ORD003,NaT,2026-05-04,2026-05-10 10:10:00,NaN,UNKNOWN
3,ORD004,NaT,2026-05-09,2026-05-10 10:15:00,NaN,UNKNOWN
4,ORD005,2026-05-05,NaT,2026-05-10 10:20:00,NaN,UNKNOWN


In [ ]:
last_watermark

Timestamp('2026-05-10 10:07:00')

In [ ]:
overlap_window = last_watermark - timedelta(minutes=5)
print(overlap_window)

overlap_df = df[
    df["updated_at"] >= overlap_window
]

overlap_df # ORD002 is in this window also even though it was updated_at 2026-05-10 10:05:00

2026-05-10 10:02:00


,order_id,order_date,delivery_date,updated_at,delivery_days,sla_status
1,ORD002,NaT,2026-05-08,2026-05-10 10:05:00,NaN,UNKNOWN
2,ORD003,NaT,2026-05-04,2026-05-10 10:10:00,NaN,UNKNOWN
3,ORD004,NaT,2026-05-09,2026-05-10 10:15:00,NaN,UNKNOWN
4,ORD005,2026-05-05,NaT,2026-05-10 10:20:00,NaN,UNKNOWN



## Why Overlap Window Important

ช่วย handle:
- late event
- delayed update
- retry scenario


## 10. Simulate Late Event

In [ ]:
late_event = {
    "order_id": "ORD999",
    "updated_at": "2026-05-10 10:03:00"
}

late_event_df = pd.DataFrame([late_event]) # แปลงให้เป็น dataframe
late_event_df

,order_id,updated_at
0,ORD999,2026-05-10 10:03:00


In [ ]:
late_event = {
    "order_id": "ORD999",
    "updated_at": "2026-05-10 10:03:00"
}

late_event_df = pd.DataFrame([late_event]) # แปลงให้เป็น dataframe

late_event_df["updated_at"] = pd.to_datetime( # แปลงข้อมูล updated_at ให้เป็นประเภท datetime
    late_event_df["updated_at"]
)

late_event_df

,order_id,updated_at
0,ORD999,2026-05-10 10:03:00



## Event Time vs Processing Time

| Concept | Meaning |
|---|---|
| Event Time | เวลาที่ event เกิดจริง |
| Processing Time | เวลาที่ pipeline โหลด |


## 11. Detect Late Event

In [ ]:
late_detected = late_event_df[
    late_event_df["updated_at"] < last_watermark
]

late_detected

,order_id,updated_at
0,ORD999,2026-05-10 10:03:00


## 12. Deduplicate Example

In [ ]:
# สร้างตัวอย่างรายการที่ซ้ำกัน คือเป็น order_id เดียวกัน แต่ถูก update สองครั้ง
duplicate_data = {
    "order_id": ["ORD001", "ORD001"],
    "updated_at": [
        "2026-05-10 10:00:00",
        "2026-05-10 10:05:00"
    ],
    "amount": [1000, 1200]
}

dup_df = pd.DataFrame(duplicate_data)

dup_df["updated_at"] = pd.to_datetime(
    dup_df["updated_at"]
)

dup_df

,order_id,updated_at,amount
0,ORD001,2026-05-10 10:00:00,1000
1,ORD001,2026-05-10 10:05:00,1200


## 13. Keep Latest updated_at

In [ ]:
# deduplicate คือ รายการที่มี order_id ซ้ำให้เอาเฉพาะรายการล่าสุดของ order_id นั้นไว้
dedup_df = dup_df.sort_values(
    by="updated_at"
).drop_duplicates(
    subset=["order_id"],
    keep="last"
)

dedup_df

,order_id,updated_at,amount
1,ORD001,2026-05-10 10:05:00,1200


## 14. Create Simple Run Metadata

In [ ]:
run_metadata = {
    "run_id": "RUN_001",
    "start_time": datetime.now(),
    "row_count": len(df),
    "status": "SUCCESS"
}

print(run_metadata)
runlog_df = pd.DataFrame(run_metadata)
runlog_df

{'run_id': 'RUN_001', 'start_time': datetime.datetime(2026, 7, 15, 6, 37, 42, 926005), 'row_count': 5, 'status': 'SUCCESS'}


ValueError: If using all scalar values, you must pass an index

## 15. Export Incremental Dataset

In [ ]:
!ls

sample_data


In [ ]:
incremental_df.to_csv(
    "incremental_orders.csv",
    index=False
)

print("Incremental Export Completed")


Incremental Export Completed



# Final Learning Outcome

ผู้เรียนควรเข้าใจ:
- Date parsing
- Invalid date handling
- SLA calculation
- Watermarking
- Incremental loading
- Overlap window
- Late event handling
- Deduplication
